In [12]:
import torch
from transformers import pipeline
import warnings

warnings.simplefilter("ignore")

### 모델 확인

In [13]:
from huggingface_hub import HfApi
api = HfApi()
info = api.model_info("klue/bert-base")
print("태스크:", info.pipeline_tag)
print("다운로드:", info.downloads)
print("라이선스:", info.card_data.get("license") if info.card_data else None)  

if info.safetensors:                      # safetensors로  올라온  모델만  있다
    print("파라미터  수:", f"{info.safetensors.total:,}")

태스크: fill-mask
다운로드: 77341
라이선스: cc-by-sa-4.0
파라미터  수: 111,243,522


### 코드로 모델 검색

In [14]:
for m in api.list_models(filter="text-classification", search="korean", sort="downloads", limit=10):
    print(f"{(m.downloads or 0):>10,} {m.id}")  

     2,913 nlp04/korean_sentiment_analysis_kcelectra
     2,202 cringepnh/koelectra-korean-sentiment
       836 matthewburke/korean_sentiment
       529 WhitePeak/bert-base-cased-Korean-sentiment
        92 Seonghaa/korean-emotion-classifier-roberta
        78 hyun5ooo/deberta-v3-base-korean-ai-detect
        48 GGARA02/kcelectra-korean-emotion
        43 2tle/korean-curse-detection
        39 beomi/korean-hatespeech-classifier
        36 dudu1021/my_doccls_korean_model


#### 감정분석

In [15]:
clf = pipeline("sentiment-analysis",model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

print(clf(["I love this phone.", "The battery dies too fast."])) 
print(clf("배송이  너무  늦어요"))

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 799.44it/s]


[{'label': 'POSITIVE', 'score': 0.999815046787262}, {'label': 'NEGATIVE', 'score': 0.9996963739395142}]
[{'label': 'POSITIVE', 'score': 0.5883867144584656}]


In [16]:
fill = pipeline("fill-mask", model="klue/bert-base") 
for r in fill("대한민국의  수도는  [MASK]이다."):
    print(r["token_str"], round(r["score"], 3))

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 467.08it/s]
[transformers] BertForMaskedLM LOAD REPORT from: klue/bert-base
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


서울 0.595
광화문 0.081
평양 0.047
부산 0.029
인천 0.026


#### 제로샷 분류 

In [17]:
zsc = pipeline("zero-shot-classification",model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")
r = zsc("손흥민이  시즌  10호  골을  터뜨렸다.", candidate_labels=["스포츠", "경제", "정 치", "IT"])
print(list(zip(r["labels"], [round(s, 3) for s in r["scores"]])))

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 1087.92it/s]


[('스포츠', 0.478), ('IT', 0.291), ('정 치', 0.128), ('경제', 0.103)]


#### 번역 (한국어 → 영어)

In [19]:
tr = pipeline("text-generation", model="Helsinki-NLP/opus-mt-ko-en")
print(tr("오늘  회의는  3시로  연기되었습니다.")[0]["translation_text"])

Loading weights: 100%|██████████| 158/158 [00:00<00:00, 23274.68it/s]
[transformers] MarianForCausalLM LOAD REPORT from: Helsinki-NLP/opus-mt-ko-en
Key                                                                 | Status     |  | 
--------------------------------------------------------------------+------------+--+-
model.encoder.layers.{0, 1, 2, 3, 4, 5}.fc1.weight                  | UNEXPECTED |  | 
model.encoder.layers.{0, 1, 2, 3, 4, 5}.self_attn.out_proj.bias     | UNEXPECTED |  | 
model.encoder.layers.{0, 1, 2, 3, 4, 5}.self_attn.q_proj.bias       | UNEXPECTED |  | 
model.encoder.layers.{0, 1, 2, 3, 4, 5}.self_attn.k_proj.bias       | UNEXPECTED |  | 
model.encoder.layers.{0, 1, 2, 3, 4, 5}.self_attn_layer_norm.weight | UNEXPECTED |  | 
model.encoder.layers.{0, 1, 2, 3, 4, 5}.self_attn.q_proj.weight     | UNEXPECTED |  | 
model.encoder.layers.{0, 1, 2, 3, 4, 5}.self_attn_layer_norm.bias   | UNEXPECTED |  | 
model.encoder.layers.{0, 1, 2, 3, 4, 5}.self_attn.out_proj.weight   |

ValueError: Unrecognized configuration class <class 'transformers.models.marian.configuration_marian.MarianConfig'> to build an AutoTokenizer.
Model type should be one of GPT2Config, Aimv2Config, AlbertConfig, AlignConfig, AriaConfig, AudioFlamingo3Config, AyaVisionConfig, BarkConfig, BartConfig, BeamConfig, BertConfig, BertGenerationConfig, BigBirdConfig, BigBirdPegasusConfig, BioGptConfig, BlenderbotConfig, BlenderbotSmallConfig, BlipConfig, Blip2Config, BridgeTowerConfig, BrosConfig, CamembertConfig, CanaryConfig, CanineConfig, ChineseCLIPConfig, ClapConfig, CLIPConfig, CLIPSegConfig, ClvpConfig, CodeGenConfig, CohereConfig, Cohere2Config, CohereAsrConfig, ColQwen2Config, ConvBertConfig, Cosmos3OmniConfig, CpmAntConfig, CTRLConfig, Data2VecAudioConfig, Data2VecTextConfig, DbrxConfig, DebertaConfig, DebertaV2Config, DiaConfig, DiffusionGemmaConfig, DistilBertConfig, DPRConfig, ElectraConfig, Emu3Config, ErnieConfig, Ernie4_5Config, Ernie4_5_MoeConfig, EsmConfig, EsmcConfig, EsmFold2C